In [1]:
import os
from dotenv import load_dotenv
load_dotenv() 

True

In [2]:
from openai import OpenAI
import os

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

try:
    response = client.models.list()
    print("Success!")
except Exception as e:
    print(e)

Success!


In [3]:
from langchain_openai  import ChatOpenAI
llm=chat=ChatOpenAI(model="gpt-5-nano", temperature=0)
llm

ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.4.7', 'langchain-openai': '1.3.2'}}, output_version=None, profile={'name': 'GPT-5 Nano', 'release_date': '2025-08-07', 'last_updated': '2025-08-07', 'open_weights': False, 'max_input_tokens': 272000, 'max_output_tokens': 128000, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': False, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True, 'tool_call_streaming': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x1110c9580>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x112334b90>, root_client=<openai.OpenAI object at 0x1110c9730>, root_async_client=<o

# *Rag impementation with pdf and my own data

# *step 1: Extracting the data from pdf

In [4]:
from langchain_community.document_loaders import PyPDFLoader
print(dir(PyPDFLoader))

['__abstractmethods__', '__class__', '__del__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__slots__', '__str__', '__subclasshook__', '__weakref__', '_abc_impl', '_is_s3_presigned_url', '_is_s3_url', '_is_valid_url', 'alazy_load', 'aload', 'lazy_load', 'load', 'load_and_split', 'source']


/var/folders/2z/pqpq9wcs0dv4jfbgkp96j3780000gn/T/ipykernel_2963/3345968884.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [5]:
pdf_path="/Users/bilal/Downloads/AiDataEngineeringPortfolio/RAG/Doc_pdf/fabric-admin.pdf"
pdf_path

'/Users/bilal/Downloads/AiDataEngineeringPortfolio/RAG/Doc_pdf/fabric-admin.pdf'

In [6]:
loader=PyPDFLoader(pdf_path)
docs=loader.load()
docs

[Document(metadata={'producer': 'Microsoft Learn PDF 1.0.25309.01', 'creator': 'Microsoft Learn', 'creationdate': '2025-12-09T19:42:08+00:00', 'title': 'fabric admin | Microsoft Learn', 'moddate': '2025-12-09T19:42:08+00:00', 'source': '/Users/bilal/Downloads/AiDataEngineeringPortfolio/RAG/Doc_pdf/fabric-admin.pdf', 'total_pages': 290, 'page': 0, 'page_label': '1'}, page_content='Tell us about your PDF experience.\nMicrosoft Fabric documentation for\nadmins\nLearn about the Microsoft Fabric admin settings, options, and tools.\nFabric in your organization\nｅOVERVIEW\nWhat is Microsoft Fabric admin?\nWhat is the admin portal?\nｂGET STARTED\nEnable Fabric for your organization\nRegion availability\nFind your Fabric home region\nｃHOW-TO GUIDE\nUnderstand Fabric admin roles\nｉREFERENCE\nGovernance documentation\nSecurity documentation\nTools and settings\nｅOVERVIEW\nAbout tenant settings\nｃHOW-TO GUIDE\nSet up git integration\nSet up item certification'),
 Document(metadata={'producer': 'Mi

In [7]:
#change metadata of the document

for i in docs:
    i.metadata={
        "source":"MFL.PDF",
        "develped_by":"Bilal"

    }

In [8]:
docs[0].metadata

{'source': 'MFL.PDF', 'develped_by': 'Bilal'}

# *step 2 : split the document into chunks and create vector store*

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter=RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)


In [10]:
chunks=splitter.split_documents(docs)
chunks

[Document(metadata={'source': 'MFL.PDF', 'develped_by': 'Bilal'}, page_content='Tell us about your PDF experience.\nMicrosoft Fabric documentation for\nadmins\nLearn about the Microsoft Fabric admin settings, options, and tools.\nFabric in your organization\nｅOVERVIEW\nWhat is Microsoft Fabric admin?\nWhat is the admin portal?\nｂGET STARTED\nEnable Fabric for your organization\nRegion availability\nFind your Fabric home region\nｃHOW-TO GUIDE\nUnderstand Fabric admin roles\nｉREFERENCE\nGovernance documentation\nSecurity documentation\nTools and settings\nｅOVERVIEW\nAbout tenant settings\nｃHOW-TO GUIDE\nSet up git integration\nSet up item certification'),
 Document(metadata={'source': 'MFL.PDF', 'develped_by': 'Bilal'}, page_content='Configure notifications\nSet up metadata scanning\nEnable content certification\nEnable service principal authentication\nConfigure Multi-Geo support\nMonitoring and management\nｅOVERVIEW\nWhat is the admin monitoring workspace?\nｐCONCEPT\nFeature usage and 

In [11]:
len(chunks) # we have 522 embeding vectors for our pdf document

522

# *step 3 : creating embedding for the chunks and storing them in vector database (we will use chroma db for this)*

In [12]:
from langchain_openai import OpenAIEmbeddings
openai_embeddings=OpenAIEmbeddings(model="text-embedding-3-small")

openai_embeddings



OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x1133c66f0>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x11337d910>, model='text-embedding-3-small', dimensions=None, deployment='text-embedding-ada-002', openai_api_version=None, openai_api_base=None, openai_api_type=None, openai_proxy=None, embedding_ctx_length=8191, openai_api_key=SecretStr('**********'), openai_organization=None, allowed_special=None, disallowed_special=None, chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None, http_async_client=None, check_embedding_ctx_length=True)

# *step 4 : store the embeddings in the exiting and local vector database*

In [13]:
from langchain_community.vectorstores import Chroma

#vectordb=Chroma.from_documents(documents=chunks, 
#                               embedding=openai_embeddings,
#                               persist_directory="/Users/bilal/Downloads/RAG/vectorDB")
#
#vectordb

vectordb=Chroma(persist_directory="/Users/bilal/Downloads/AiDataEngineeringPortfolio/RAG/vectorDB", embedding_function=openai_embeddings)


/var/folders/2z/pqpq9wcs0dv4jfbgkp96j3780000gn/T/ipykernel_2963/2801501404.py:9: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectordb=Chroma(persist_directory="/Users/bilal/Downloads/AiDataEngineeringPortfolio/RAG/vectorDB", embedding_function=openai_embeddings)


In [14]:
#now add the vector in data base
vectordb.add_documents(chunks)

['d83a11a6-904b-4d76-9f33-885ad94fa8e9',
 '27eeea57-c7e5-4e80-804a-17dc7d8e01ef',
 '375496c3-aee9-4b5e-a471-4c8b306dc4b1',
 '9b815047-06d1-44b5-a92b-83cf78f32e7b',
 'd108de9d-dad6-4c12-b46f-a3951885c537',
 'f01ea5e1-5a8d-44b1-8f3b-982acaa28596',
 '2afe4d80-cb99-4720-81e5-b3ba3cb30991',
 'caf0ff7e-1a39-4826-9662-c4be881b54b5',
 'ebcb063a-2c1e-4503-b793-976829a8a0ca',
 '5300b87e-f12c-4baf-85ea-3d17de719c45',
 '842ae0ca-e742-4171-b3e5-5574aa18253e',
 '989710a7-f8ac-4cdc-b310-67a496997fe3',
 'eed81230-3627-4aee-87e0-51a3762e89af',
 '0e53c191-f34b-4eac-b228-1d2821e4e117',
 'ade34c0a-b0e7-4eb7-bd4e-41f2cf8cc1f0',
 '1ac837ca-e43f-4405-a315-13fceb128bd0',
 '9c5f9a75-11b6-4e2e-9c8c-56f351583c41',
 '76b44a02-05b0-4896-9247-aaf540da4ad8',
 'b6588a5f-9a1b-42c8-8d9f-8a8fb924fe9e',
 '99fa226b-9d68-46ef-bf84-0d4f9689d6e0',
 'f0d8ea4f-9baa-4c76-9605-e9545d250840',
 '613af469-04c8-417a-9d61-e6c4e9c8dfa0',
 '958f3464-2e03-44f9-8e5d-46797a21fe43',
 '837cb79c-8bdd-436f-8a80-6a25f710c1d5',
 '967861bf-d5d6-

# *step 5 : Semantic search*

In [15]:
#ask the question to the vector data base
vectordb.similarity_search("What is microsoft fabric admin?", k=3)

[Document(metadata={'develped_by': 'Bilal', 'source': 'MFL.PDF'}, page_content='What is Microsoft Fabric admin?\n07/01/2024\nMicrosoft Fabric admin is the management of the organization-wide settings that control how\nMicrosoft Fabric works. Users that are assigned to admin roles configure, monitor, and\nprovision organizational resources. This article provides an overview of admin roles, tasks, and\ntools to help you get started.\nThere are several roles that work together to administer Microsoft Fabric for your organization.\nMost admin roles are assigned in the Microsoft 365 admin portal or by using PowerShell. The\ncapacity admin roles are assigned when the capacity is created. To learn more about each of\nthe admin roles, see About admin roles. To learn how to assign admin roles, see Assign admin\nroles.\nThis section lists the Microsoft 365 admin roles and the tasks they can perform.\nGlobal administrator\nUnlimited access to all management features for the organization\nAssign r

# *Resuse the vectore database*

In [16]:

from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings

embeddings_model_2=OpenAIEmbeddings(model="text-embedding-3-small")


vectordb_persisted=Chroma(persist_directory="/Users/bilal/Downloads/AiDataEngineeringPortfolio/RAG/vectorDB", embedding_function=embeddings_model_2)

In [17]:
vectordb_persisted.similarity_search("What is fabric onelake?", k=3)

[Document(metadata={'producer': 'Microsoft Learn PDF 1.0.25309.01', 'total_pages': 382, 'creationdate': '2025-12-12T17:01:10+00:00', 'version': '2025', 'owner': 'Microsoft', 'page': 333, 'document_name': 'Fabric Guide', 'page_label': '334', 'department': 'Data Engineering', 'moddate': '2025-12-12T17:01:10+00:00', 'source': '/Users/bilal/Downloads/AiDataEngineeringPortfolio/RAG/Doc_pdf/fabric-onelake.pdf', 'title': 'fabric onelake | Microsoft Learn', 'creator': 'Microsoft Learn'}, page_content="Fabric capacity and OneLake\nconsumption\nArticle• 12/26/2024\nYou only need one capacity to drive all your Microsoft Fabric experiences, including\nMicrosoft OneLake. Keep reading if you want a detailed example of how OneLake\nconsumes storage and compute.\nOneLake comes automatically with every Fabric tenant and is designed to be the single\nplace for all your analytics data. All the Fabric data items are prewired to store data in\nOneLake. For example, when you store data in a lakehouse or war

Existing Vector Database

↓

Load Existing Database

↓

Read New PDF

↓

Chunk

↓

Embedding

↓

Append New Vectors

↓

Save Automatically

↓

Ask Questions from BOTH PDFs

Existing DB

+

New PDF

↓

Add New Vectors

Mini Project 2
Enterprise Knowledge Base Updater
Scenario

Initially your company has Fabric-OneLake.pdf

Later another team uploads

Fabric-Admin.pdf

Instead of rebuilding the database,

you simply append new documents.



             Existing Vector DB
                     │
                     ▼

           Load Existing Chroma

                     │

         New PDF (Fabric Admin)

                     │

               PyPDFLoader

                     │

                  Documents

                     │

              Metadata Update

                     │

                  Chunking

                     │

             OpenAI Embeddings

                     │

             Add Documents()

                     │

             Updated Vector DB

                     │

              User Question

                     │

              Similarity Search

                     │

                GPT Answer

One File Mini Project
What This Project Teaches

This project introduces a new concept compared to your previous one:

Previous Project	This Project
Create a brand-new vector database	Load an existing vector database
Store the first PDF	Add a second PDF
Chroma.from_documents()	Chroma(...) + add_documents()
Initial ingestion	Incremental ingestion
One knowledge source	Growing knowledge base

A good answer is:

Chroma.from_documents() is used when you are creating a new vector database. It embeds the documents and stores them in a new collection.
add_documents() is used when the vector database already exists. It embeds only the new documents and appends them to the existing collection without rebuilding everything.

In [20]:
# ==========================================================
# MINI PROJECT 2
# Update Existing Enterprise Knowledge Base
# ==========================================================


# ==========================================================
# STEP 1 : Load Environment
# ==========================================================

import os
from dotenv import load_dotenv

load_dotenv()

print(
    "API Loaded:",
    bool(os.getenv("OPENAI_API_KEY"))
)


# ==========================================================
# STEP 2 : Import Libraries
# ==========================================================

from langchain_community.document_loaders import PyPDFLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_openai import (
    OpenAIEmbeddings,
    ChatOpenAI
)

from langchain_community.vectorstores import Chroma


# ==========================================================
# STEP 3 : Paths
# ==========================================================

PDF_PATH = "/Users/bilal/Downloads/AiDataEngineeringPortfolio/RAG/Doc_pdf/fabric-admin.pdf"

VECTOR_DB = "/Users/bilal/Downloads/AiDataEngineeringPortfolio/RAG/vectorDB"


# ==========================================================
# STEP 4 : Load Existing Vector Database
# ==========================================================

print("\nLoading Existing Vector Database...")

embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

vector_db = Chroma(

    persist_directory=VECTOR_DB,

    embedding_function=embedding_model

)

print("Vector Database Loaded")


# ==========================================================
# STEP 5 : Load New PDF
# ==========================================================

print("\nLoading New PDF...")

loader = PyPDFLoader(PDF_PATH)

documents = loader.load()

print("Pages :", len(documents))


# ==========================================================
# STEP 6 : Update Metadata
# ==========================================================

print("\nUpdating Metadata...")

for doc in documents:

    doc.metadata = {

        "document_name":"Fabric Admin",

        "department":"Administration",

        "uploaded_by":"Bilal",

        "version":"2025"

    }

print(documents[0].metadata)


# ==========================================================
# STEP 7 : Chunking
# ==========================================================

print("\nCreating Chunks...")

splitter = RecursiveCharacterTextSplitter(

    chunk_size=1000,

    chunk_overlap=100

)

chunks = splitter.split_documents(documents)

print("Chunks :", len(chunks))


# ==========================================================
# STEP 8 : Add New Documents
# ==========================================================

print("\nAdding New Chunks To Existing Database...")

vector_db.add_documents(chunks)

print("Database Updated Successfully")


# ==========================================================
# STEP 9 : Ask Question
# ==========================================================

question = input("\nAsk Question : ")


# ==========================================================
# STEP 10 : Search
# ==========================================================

results = vector_db.similarity_search_with_score(

    question,

    k=3

)

context = ""

sources = []

print("\nRetrieved Chunks")

for index,(doc,score) in enumerate(results):

    print("\n======================")

    print(f"Chunk {index+1}")

    print("======================")

    print("Similarity :", score)

    print()

    print(doc.page_content[:400])

    print()

    print(doc.metadata)

    context += "\n\n" + doc.page_content

    sources.append(doc.metadata)


# ==========================================================
# STEP 11 : Prompt
# ==========================================================

prompt = f"""

You are an Enterprise AI Assistant.

Answer only from the context.

Context

{context}

Question

{question}

"""


# ==========================================================
# STEP 12 : LLM
# ==========================================================

llm = ChatOpenAI(

    model="gpt-5-nano",

    temperature=0

)

response = llm.invoke(prompt)


# ==========================================================
# STEP 13 : Final Answer
# ==========================================================

print("\n======================")
print("FINAL ANSWER")
print("======================")

print(response.content)

print("\n======================")
print("SOURCES")
print("======================")

for source in sources:

    print(source)

API Loaded: True

Loading Existing Vector Database...
Vector Database Loaded

Loading New PDF...
Pages : 290

Updating Metadata...
{'document_name': 'Fabric Admin', 'department': 'Administration', 'uploaded_by': 'Bilal', 'version': '2025'}

Creating Chunks...
Chunks : 522

Adding New Chunks To Existing Database...
Database Updated Successfully

Retrieved Chunks

Chunk 1
Similarity : 1.3506890535354614

Microsoft Fabric security
Microsoft Fabric is a software as a service (SaaS) platform that offers a complete security
package. Fabric removes the cost and responsibility of maintaining your security
solution and transfers it to the cloud. With Fabric, you can use the expertise and
resources of Microsoft to keep your data secure, patch vulnerabilities, monitor threats,
and comply with regulations.


{'develped_by': 'Bilal', 'source': 'MFL.PDF'}

Chunk 2
Similarity : 1.3506890535354614

Microsoft Fabric security
Microsoft Fabric is a software as a service (SaaS) platform that offers a comp